# 00 — Environment check (AMD MI300X / ROCm)

Run **first** in each daily GPU window. Verifies ROCm + bf16, installs deps, logs into W&B,
and sets `PERSIST_DIR` (the mounted folder where data + checkpoints survive between sessions).

In [1]:
# Install dependencies (torch/ROCm already in the AMD image). Safe to re-run.
# Comment out once the persistent env has them.
%pip install -q albumentations torchstain scikit-image timm transformers accelerate wandb gradio opencv-python-headless seaborn scikit-learn
print("deps installed")



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
deps installed


In [2]:
import torch, platform
print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("ROCm/HIP    :", getattr(torch.version, "hip", None))
print("CUDA build  :", torch.version.cuda)
cuda = torch.cuda.is_available()
print("GPU available:", cuda)
if cuda:
    print("Device      :", torch.cuda.get_device_name(0))
    print("bf16 native :", torch.cuda.is_bf16_supported())
    free, total = torch.cuda.mem_get_info()
    print(f"HBM         : {total/1e9:.0f} GB total, {free/1e9:.0f} GB free")
else:
    print("WARNING: no GPU visible — fine for authoring, but training needs the MI300X session.")


Python      : 3.12.11
PyTorch     : 2.8.0+gitb2fb688
ROCm/HIP    : 7.0.51831-a3e329ad8
CUDA build  : None
GPU available: True
Device      : 
bf16 native : True
HBM         : 206 GB total, 145 GB free


In [3]:
import os
from pathlib import Path

# Point this at the AMD *persistent / mounted* folder so data + checkpoints survive restarts.
PERSIST_DIR = Path(os.environ.get("PERSIST_DIR", "/workspace/shared/ft004"))
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
for sub in ("data", "checkpoints", "outputs"):
    (PERSIST_DIR / sub).mkdir(exist_ok=True)
os.environ["PERSIST_DIR"] = str(PERSIST_DIR)
print("PERSIST_DIR =", PERSIST_DIR)
print("Free disk (GB):", round(__import__('shutil').disk_usage(PERSIST_DIR).free / 1e9, 1))


PERSIST_DIR = /workspace/shared/ft004
Free disk (GB): 14.0


In [4]:
# Weights & Biases login. Needs an API key on this machine.
# If network egress is blocked, switch to offline mode instead:
#   import os; os.environ["WANDB_MODE"] = "offline"
import wandb
try:
    wandb.login()           # prompts for key, or uses WANDB_API_KEY env var
    print("wandb: logged in")
except Exception as e:
    os.environ["WANDB_MODE"] = "offline"
    print(f"wandb login failed ({e}); falling back to WANDB_MODE=offline")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Currently logged in as: thunderhill4 (thunderhill4-tcs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: logged in
